# $h_y$ sweep — first-order transition vs $L$
Pure-$Y$ field ($h_x=h_z=0$), $h_y\in[0.80,1.20]$, CNN ansatz. Converged runs only (skip NaN / no-`.mpack`). The transition shows as a **discontinuity** in $\langle\sigma^y\rangle$ / $\langle B_p\rangle$.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
NQS = Path("../results/nqs"); FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

In [ ]:
ARM = "cnn"

def sweep(L):                              # {hy: data} for converged runs only
    out = {}
    for p in sorted(NQS.glob(f"G-equiv_1_L{L}_hx0.00_hy*_{ARM}.json")):
        if not p.with_suffix(".mpack").exists():      # skip walltime-killed partials
            continue
        d = json.load(open(p))
        if not np.isfinite(complex(d["energy"][-1]).real):   # skip diverged (NaN) runs
            continue
        out[float(re.search(r"hy([\d.]+)_", p.name).group(1))] = d
    return out

sy = lambda d: np.mean(d["order_params"]["magnetization_Ymean"][-1])  # <sigma^y> order param
bp = lambda d: d["order_params"]["Bp_mean"][-1]                       # <B_p> plaquette stabilizer
en = lambda d: complex(d["energy"][-1]).real                         # converged energy

Ls = [L for L in (4, 6, 8) if sweep(L)]
data = {L: sweep(L) for L in Ls}
cols = dict(zip(Ls, plt.cm.viridis(np.linspace(0, 0.75, len(Ls)))))

def curve(L, fn):
    hy = np.array(sorted(data[L]))
    return hy, np.array([fn(data[L][h]) for h in hy])

{L: sorted(data[L]) for L in Ls}          # what got loaded

In [ ]:
# order parameters vs h_y — the first-order jump
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4), sharex=True)
for L in Ls:
    hy, y = curve(L, sy); a1.plot(hy, y, "o-", color=cols[L], ms=4, lw=1.3, label=f"L={L}")
    hy, y = curve(L, bp); a2.plot(hy, y, "o-", color=cols[L], ms=4, lw=1.3, label=f"L={L}")
for ax in (a1, a2):
    ax.axvline(1.0, color="k", ls=":", lw=1, alpha=0.6)               # theory h_c=1
    ax.set_xlabel("$h_y$")
a1.set(ylabel=r"$\langle\sigma^y\rangle$", title="order parameter")
a2.set(ylabel=r"$\langle B_p\rangle$", title="plaquette stabilizer")
a1.legend()
fig.tight_layout()
fig.savefig(FIG / "hy_orderparams_vs_L.png", dpi=300, bbox_inches="tight")

In [ ]:
# derivative sharpens the discontinuity: peak in d<sigma^y>/dh_y locates h_c(L)
fig, ax = plt.subplots(figsize=(5.5, 4))
for L in Ls:
    hy, y = curve(L, sy)
    ax.plot(hy, np.gradient(y, hy), "o-", color=cols[L], ms=4, lw=1.3, label=f"L={L}")
    print(f"L={L}: steepest d<sy>/dhy near h_y={hy[np.argmax(np.gradient(y, hy))]:.2f}")
ax.axvline(1.0, color="k", ls=":", lw=1, alpha=0.6, label="$h_c=1$ (theory)")
ax.set(xlabel="$h_y$", ylabel=r"$d\langle\sigma^y\rangle/dh_y$",
       title="susceptibility — peak locates the transition")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "hy_dsy_vs_L.png", dpi=300, bbox_inches="tight")